<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_83_cnns_deep_dive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🖼️ **Module 83 — CNNs Deep Dive (LeNet → ResNet → EfficientNet → YOLO / DETR)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 11 · Foundations Backfill


# Module 83 — CNNs Deep Dive

> Transformers (M19–M24) and VLMs (M65) eat all the headlines, but the **convolutional neural network** is still under almost every production computer-vision pipeline: defect detection, medical imaging, autonomous driving, OCR, satellite imagery, mobile-app photo features, and as the *image encoder* of every VLM. **This module is the depth pass on CNNs the course referenced but never built standalone** — from the 1998 original to 2024-era detectors.
>
> By the end you can read every CV paper, pick the right backbone, fine-tune a pretrained model with one notebook, and decide between YOLO, Mask-RCNN, DETR, and SAM for any vision task.

### What you'll cover
1. Why convolutions — the inductive bias that makes them win on images
2. The convolution mechanics — kernels, stride, padding, channels, receptive field
3. **LeNet-5 (1998)** — the original; runnable in 50 lines
4. **AlexNet (2012)** — ReLU + GPU + dropout
5. **VGG (2014)** — stack 3×3, go deeper
6. **Inception / GoogLeNet (2014)** — parallel branches + 1×1 bottlenecks
7. **ResNet (2015)** — skip connections; the unlock for very deep nets
8. **EfficientNet (2019)** — compound scaling; the parameter-efficient family
9. **MobileNet (2017+)** — depthwise-separable convs for phones
10. **ConvNeXt (2022)** — modern CNN that beats early ViTs
11. **Detection + segmentation** — YOLO · Faster R-CNN · DETR · Mask R-CNN · SAM
12. The CNN ↔ ViT relationship + what to pick in 2026


## 1 · Why convolutions

A **fully-connected layer** treats a 224×224 RGB image as a 150 528-dim vector. The first layer alone needs ~150 528 × hidden units of parameters and *throws away* the spatial structure entirely.

A **convolution** does three things instead:
- **Local receptive field** — each output pixel looks at a small `k × k` window of the input, not the whole image.
- **Weight sharing** — the same kernel slides across every position, so the same feature detector applies everywhere.
- **Translation equivariance** — if the input shifts, the activations shift in lockstep. A face two pixels to the left fires the same "eye" detector.

Result: orders of magnitude fewer parameters, faster training, and **the right prior for natural images**. That's the inductive bias.

## 2 · The convolution operation

In [ ]:
import torch, torch.nn as nn

# A single 2-D convolution: 3 input channels, 16 output channels, 3x3 kernel
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
x = torch.randn(8, 3, 32, 32)        # (B, C_in, H, W) — 8 images, 32x32, RGB
y = conv(x)
print("input :", x.shape)              # torch.Size([8, 3, 32, 32])
print("output:", y.shape)              # torch.Size([8, 16, 32, 32])
print("params:", sum(p.numel() for p in conv.parameters()))   # 3*16*3*3 + 16

**The five knobs you'll tune 1000 times:**

| Knob | What it does |
|---|---|
| `kernel_size` | window the kernel sees per output (3 or 5 in practice) |
| `stride` | how many pixels the kernel hops per step (2 halves the resolution) |
| `padding` | pixels added at the borders (`padding=1` with `kernel=3` keeps H/W) |
| `dilation` | spacing between kernel taps — for atrous convs (semantic seg) |
| `groups` | partition channels — `groups=in_channels` gives **depthwise** convs |

**Receptive field** = how many input pixels one output pixel depends on. Stacking three `3×3` convs gives a 7×7 receptive field with **9× fewer params** than one 7×7 conv. That's the VGG insight.

## 3 · LeNet-5 (1998) — the original

Yann LeCun's 1998 architecture for handwritten-digit recognition. Two conv blocks + two FC layers + a softmax over 10 classes.

In [ ]:
class LeNet5(nn.Module):
    """~60 000-param CNN — runs in ~30 seconds on MNIST CPU."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, padding=2), nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Conv2d(6, 16, kernel_size=5),           nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120), nn.Tanh(),
            nn.Linear(120, 84),         nn.Tanh(),
            nn.Linear(84, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

m = LeNet5()
print(m)
print("params:", sum(p.numel() for p in m.parameters()))

**What LeNet got right** (and most modern CNNs still inherit): **conv → activation → spatial-downsample**, repeated. **What it got wrong by 2012:** tanh activation (slow), average pooling (lossy), too shallow.

## 4 · AlexNet (2012) — three changes that started the deep-learning era

Krizhevsky-Sutskever-Hinton's ImageNet winner. Five conv layers + three FC layers, 60M parameters. **Three changes** turned LeNet from a toy into a category-winner:

| Change | Why it mattered |
|---|---|
| **ReLU activation** | non-saturating gradient → trains much deeper nets |
| **GPU training** | first big CV model trained on consumer GPUs |
| **Dropout** | random-zero half the activations during training → strong regulariser |
| **Local response normalisation** | (later replaced by BatchNorm) |
| **Data augmentation** | random crops + horizontal flips |

AlexNet didn't introduce a new architecture so much as prove that **scaling + tricks** transforms CNNs. Its 16.4 % ImageNet error vs the previous SOTA's 25.8 % is what kicked off the modern era.

## 5 · VGG (2014) — go deep, stack 3×3

VGG-16 and VGG-19 stripped the architecture down to **one block type**: stacked `3×3` convs followed by `2×2` max-pool. Two insights:

- **Three stacked `3×3` convs = one `7×7` receptive field with 1/3 the params**.
- **Doubling channels every time you halve resolution** keeps compute roughly constant per block.

```
   3×3 conv, 64  →  3×3 conv, 64  →  2×2 pool
   3×3 conv, 128 →  3×3 conv, 128 →  2×2 pool
   3×3 conv, 256 →  3×3 conv, 256 →  3×3 conv, 256 →  2×2 pool
   3×3 conv, 512 →  3×3 conv, 512 →  3×3 conv, 512 →  2×2 pool
   3×3 conv, 512 →  3×3 conv, 512 →  3×3 conv, 512 →  2×2 pool
   FC 4096       →  FC 4096       →  FC 1000 (classes)
```

VGG-16 has **138M params** and is still the canonical "image-encoder" reference in style-transfer and perceptual-loss research.

## 6 · Inception / GoogLeNet (2014) — parallel branches

Google's ImageNet 2014 winner. Instead of stacking, **run multiple kernel sizes in parallel** and concatenate the outputs. The **Inception block**:

```
                    ┌── 1×1 conv ──────────────────►─┐
   input ── split ──┼── 1×1 conv → 3×3 conv ──────►─┼── concat ─→ next
                    ├── 1×1 conv → 5×5 conv ──────►─│
                    └── 3×3 max-pool → 1×1 conv ──►─┘
```

Two ideas that still matter:
- **1×1 convs** cheaply mix channels (and were used to *reduce* channels before expensive 3×3 / 5×5 ops — the **bottleneck**).
- **Auxiliary classifiers** plugged into intermediate layers added gradient signal — pre-figured ResNet's idea.

GoogLeNet has only **6.8M params** — 20× fewer than VGG-16 — yet matched it. Parameter efficiency mattered.

## 7 · ResNet (2015) — the breakthrough

He et al. showed that *naively* stacking more layers makes training **harder**, not easier. Their fix: **skip connections**.

```
   x ─────────────────────────────┐
       ┌─ 3×3 conv → BN → ReLU ─┐ │
       │     ↓                    │ │
       │ 3×3 conv → BN           │ │
       └────────────────────────┘ │
                                  ▼
                                  + ─→ ReLU
```

The block learns the **residual** `F(x) = output − x`, not the full transformation. If the optimal function is roughly identity, gradients flow through the skip even when `F` is zero. Result: training **152 layers** without degradation.

**Three additional ResNet ingredients standard today:**
- **BatchNorm** after every conv (we replaced it with LayerNorm in ConvNeXt below).
- **Bottleneck blocks** (1×1 → 3×3 → 1×1) for deeper variants — same idea as Inception.
- **Stride-2 conv** for downsampling (replaces max-pool in modern variants).

ResNet-50 is still the **most-loaded backbone in production CV** in 2026. Every fine-tuning notebook starts with `torchvision.models.resnet50(weights="DEFAULT")`.

In [ ]:
# Loading a pre-trained ResNet-50 and replacing the final head — the classic
# CV transfer-learning recipe, alive and well.
from torchvision.models import resnet50, ResNet50_Weights
m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
m.fc = nn.Linear(m.fc.in_features, 10)        # custom 10-way classifier
print("trainable params (head only):", sum(p.numel() for p in m.fc.parameters()))
print("backbone params (frozen):", sum(p.numel() for p in m.parameters() if not p.requires_grad))

## 8 · EfficientNet (2019) — compound scaling

Tan & Le's insight: depth, width, and input resolution should all scale **together**, by a principled formula:

```
   depth        d = α^φ        (more layers)
   width        w = β^φ        (more channels per layer)
   resolution   r = γ^φ        (bigger input image)

   subject to:  α · β² · γ² ≈ 2   (each unit of φ roughly doubles compute)
```

A baseline architecture (**EfficientNet-B0**, ~5M params) plus values of `α, β, γ` found by grid search yields EfficientNet-B1 … B7 — a family with the **best accuracy/FLOPs frontier** of the late 2010s.

The 2021 **EfficientNetV2** added Fused-MBConv blocks for faster training. Still a top pick today when you need to deploy a strong image classifier on **moderate GPU/CPU hardware**.

## 9 · MobileNet (2017+) — depthwise separable convs for phones

Google's mobile-first CNN. The core trick: a normal `3×3` conv mixes both **spatial** and **channel** info. Split that into two cheaper ops:

```
   normal 3×3 conv:           cost ≈ k·k·C_in·C_out
   depthwise-separable:
       1) depthwise 3×3 conv  (per channel, no mixing)   ≈ k·k·C_in
       2) pointwise  1×1 conv (mix channels, no spatial) ≈ C_in·C_out
   ratio of (1+2) to normal:  ~ 1/C_out + 1/(k·k)
```

For `k=3, C_out=64` that's **~12× fewer multiplies**. MobileNet-V1 → V2 (inverted residuals) → V3 (NAS + h-swish) refined the family. Sister architectures: **ShuffleNet**, **GhostNet**, **Apple MobileOne**. All inhabit the same "millisecond-per-image on a phone" tier.

## 10 · ConvNeXt (2022) — modernised CNN, post-ViT

When Vision Transformers (ViT, 2020) started beating CNNs on ImageNet, Liu et al. asked: **can a CNN with similar design choices match a ViT?** Yes — they updated ResNet step-by-step with:

| Modernisation | Borrowed from |
|---|---|
| Patchify stem (4×4 stride-4 conv replaces conv+pool) | ViT |
| Inverted bottleneck (96 → 384 → 96 channels) | MobileNet-V2 |
| Larger 7×7 depthwise kernels | Swin Transformer |
| **LayerNorm replacing BatchNorm** | Transformers |
| **GELU activation replacing ReLU** | Transformers |
| Fewer activation + norm layers | Transformers |

The result, **ConvNeXt** (and ConvNeXt-V2), matched the strongest ViT variants of 2022 on ImageNet with **lower latency**. Tells you the architecture race is more about *design choices* than *convs vs attention*.

> 🟡 **Where CNNs live in 2026.** As production backbones for classification, detection, segmentation, OCR — and as the **patch embedder** at the front of many VLMs (M65). When latency or efficiency matters, CNNs still win; when scaling laws matter most, ViT + transformers do.

## 11 · Detection + segmentation

Image classification = one label per image. Two harder tasks dominate production CV:

### Object detection — "what + where + bounding box"

| Family | Idea | When |
|---|---|---|
| **R-CNN → Fast → Faster R-CNN** (2014–15) | propose regions, classify each | accuracy-critical; offline batch |
| **YOLO v1 → v8 → v10 (2024)** | dense one-shot grid; predict bbox+class per anchor | real-time inference (the most common production choice) |
| **SSD** (2016) | single-shot, multi-scale | mobile / embedded |
| **RetinaNet** (2017) | one-shot with focal loss for class imbalance | small-object scenarios |
| **DETR** (2020) + variants (Deformable-DETR, DINO-DETR) | transformer; no NMS; set prediction | clean architecture; SOTA accuracy 2023+ |
| **YOLO-World / OWL-ViT / Grounding-DINO** | open-vocabulary detection with text prompts | "detect anything you can describe" |

```python
# Production-default object detection in 5 lines (Ultralytics YOLO v8/v10)
# !pip install ultralytics
from ultralytics import YOLO
model = YOLO("yolov10n.pt")                    # 'n' = nano; 't', 's', 'm', 'l', 'x' bigger
results = model("path/to/image.jpg")            # also accepts URLs, video, webcam
for r in results: r.show()
```

### Segmentation — "label every pixel"

| Family | Idea | When |
|---|---|---|
| **FCN / U-Net** (2015) | encoder-decoder with skip connections | medical, satellite imagery |
| **DeepLab** (2017+) | atrous (dilated) convs + ASPP | urban / semantic |
| **Mask R-CNN** (2017) | Faster R-CNN + mask head | instance segmentation |
| **Mask2Former** (2022) | transformer-based universal seg | panoptic |
| **SAM / SAM 2** (2023–24) | "segment anything"; promptable | zero-shot interactive seg |

**Production rule of thumb 2026:** **YOLO v10** for general object detection, **SAM 2** for interactive/edge segmentation, **DETR** family if you can spare the compute and want clean training.

## 12 · CNNs ↔ ViT in 2026 — what to pick

```
   small image (< 224²)              MobileNet / EfficientNet-Lite
   accuracy-first classification     ConvNeXt or ViT-L (close)
   detection production              YOLOv10 / RT-DETR
   detection accuracy-frontier       DINO-DETR / Co-DETR
   semantic segmentation             Mask2Former / SegFormer
   instance / interactive seg        Mask R-CNN / SAM 2
   open-vocabulary                   OWL-ViT / Grounding-DINO / YOLO-World
   image encoder for VLMs            ViT (most common); CNN backbones in MobileVLM
   mobile / on-device                MobileNet / GhostNet / Apple MobileOne
   ultra-large-scale pre-training    ViT (CLIP, SigLIP, DINOv2)
```

### The two crossovers worth knowing

1. **CNN backbones inside transformers.** Many 2024-26 detectors (RT-DETR, EfficientDet-V2) use a ResNet / ConvNeXt **backbone** and a transformer **head** — best of both.
2. **Hybrid architectures.** CoAtNet, MaxViT, CvT and Swin-Transformer alternate conv + attention. They consistently win on small-data regimes where pure ViT overfits.

## 13 · Production CV patterns

| Pattern | When |
|---|---|
| **Pretrained backbone + new head** | almost always; `timm` + `torchvision` ship hundreds |
| **`torch.compile`** | 1.3–2× training speedup with minimal code change |
| **ONNX export → ONNX Runtime / TensorRT (M71)** | for cross-language / GPU-edge serving |
| **TFLite / Core ML / GGUF** | mobile, embedded (M90) |
| **Augmentations: Albumentations / Kornia** | the modern, production-grade libraries |
| **Mixed precision (`bf16` / `fp16`)** | 2× throughput, virtually free |
| **Distillation** (M88 preview) | shrink a big ViT into a small CNN for serving |
| **CLIP-style image encoders** | when you need image ↔ text alignment, not class labels |

```python
# A complete production-shape CV fine-tuning loop (PyTorch + timm + Albumentations)
fine_tune_sketch = '''
import timm, albumentations as A, torch
from torch.utils.data import DataLoader, Dataset

train_tf = A.Compose([
    A.RandomResizedCrop(224, 224), A.HorizontalFlip(),
    A.Normalize(), A.pytorch.ToTensorV2(),
])

model = timm.create_model("convnext_tiny", pretrained=True, num_classes=NUM_CLASSES)
model = torch.compile(model)                         # 1.5x train speedup
opt   = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

for epoch in range(EPOCHS):
    for x, y in train_loader:
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            loss = nn.functional.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    sched.step()
'''
print(fine_tune_sketch)

**Three tools to know by name:**
- **`timm`** (Ross Wightman / HF) — hundreds of pretrained image models, one API.
- **`torchvision.models`** — the curated PyTorch subset.
- **Ultralytics** — `pip install ultralytics`; YOLO + utilities + export to ONNX/TensorRT in one CLI.

For the *interview*: be able to sketch ResNet, explain skip connections, name 3 detection families and when each wins.

## ✅ Recap
- Convolutions encode **locality**, **weight sharing**, and **translation equivariance** — the right prior for natural images.
- **LeNet → AlexNet → VGG → Inception → ResNet → EfficientNet → MobileNet → ConvNeXt** is the architectural arc.
- The single most important idea is **skip connections** — without ResNet we don't have networks deeper than ~20 layers.
- **EfficientNet** taught us to scale depth + width + resolution **together** with a principled rule.
- **MobileNet** depthwise-separable convs are the on-device default; **ConvNeXt** is the modern desktop/server backbone.
- **Detection**: YOLO v10 for production speed, DETR family for accuracy, SAM 2 for interactive seg.
- CNN backbones are still under almost every production CV pipeline and most VLM image encoders (M65).

Next: **M84 — RNN / LSTM / GRU Deep Dive**.
